In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from vector_search.embedder import Embedder

embed = Embedder(path=r"..\models\Xenova\all-MiniLM-L6-v2")

# Q1. Embedding a query

In [8]:
q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)

In [9]:
v1[0]

np.float64(-0.02058203437252893)

# Q2. Cosine similarity

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
target = next(d for d in documents if d['filename'].endswith('07-sqlitesearch-vector.md'))
v_target = embed.encode(target['content'])
v_target.dot(v1)

import numpy as np
np.dot(v1, v_target)

np.float64(0.36107026789538205)

# Q3. Chunking and search by hand

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

import numpy as np
texts = [c['content'] for c in chunks]
X = np.array(embed.encode_batch(texts))

scores = X.dot(v1)
idx = int(np.argmax(scores))
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

In [13]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=[])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
q_vec = embed.encode(query)

results = vindex.search(q_vec, num_results=1)
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

# Q5. Text search vs vector search

In [14]:
from minsearch import Index

tindex = Index(text_fields=['content'])
tindex.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

t_results = tindex.search(query, num_results=5)
t_files = {r['filename'] for r in t_results}

v_results = vindex.search(embed.encode(query), num_results=5)
v_files = {r['filename'] for r in v_results}

v_files - t_files

{'02-vector-search/lessons/08-pgvector.md'}

# Q6. Hybrid search


In [15]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query = "How do I give the model access to tools?"

vector_results = vindex.search(embed.encode(query), num_results=5)
text_results = tindex.search(query, num_results=5)

fused = rrf([vector_results, text_results])
fused[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'